# 第0阶段之v2

## 8. 修正：重建V2 并冻结

机器学习项目，不能做完实验凭感觉写结论，要把所有规则写成一张"标准说明书（口径规格表）"，交给 AI / 代码去自动执行、自动检查，保证以后任何人复现、迭代都不走样。

很多同学做毕设、做机器学习项目，踩一个大坑：代码写得乱糟糟，清洗、筛选、打标签的规则零散藏在代码里面，自己当时知道怎么选样本，过两周就忘了；换一个人跑代码，稍微改一行，样本就变了，模型结果直接变，谁也说不清差异是模型导致的，还是数据筛选规则偷偷变了。这张表，就是把咱们前面所有复盘、踩过的坑，全部固化成白纸黑字的硬性规则，也就是数据契约。

### 例如：通过AI实现上面的规范

按下面的规格表，从 incident.db 重新构建训练集和验证集。这是一个新的 Notebook，不是修改已有代码——按表重写。

【环境】Python 3.12，pandas 3.x；数据库 ../data/incident.db；产物写到 ../output/v2/（不存在就建）。

三张表：
- residents(resident_id, gender, birth_date, education, income_level, district, enroll_date)
- visits(visit_id, resident_id, visit_date, created_at, known_diabetes, chronic_mgmt_tag, height_cm, weight_kg, waist_cm, sbp, dbp, smoking_status, drinking_freq, exercise_min_week, sleep_hours, veg_fruit_daily, family_history_dm, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained)
- lab_results(lab_id, resident_id, sample_date, report_date, fpg_mmol_l, hba1c_pct)

已知的写法问题：gender 里有 male、female、男、女；smoking_status 有大小写不一、首尾空格，以及 从不、吸烟、已戒；drinking_freq 有 不喝、偶尔、经常。

【规格表】
- 筛查人群：只保留 known_diabetes == 0 的随访；该列不进输出
- 标签：随访日起 90 天内（含当天）的化验，取采样最早的一条（同日再按 lab_id）；空腹血糖 >= 7.0 或糖化 >= 6.5 为 1，否则为 0
- 没化验：窗口内没有已出报告的化验 → 该随访不进数据集（不是阴性）
- 训练快照日：2024-06-30，训练集只能用这一天已经出报告的化验
- 训练随访最晚：2024-06-30 −（90 + 30）天 = 2024-03-02
- 验证区间：2024-07-01 ～ 2024-12-31（标签用导出日 2025-06-30 已出的报告）
- 隔离期：2024-03-03 ～ 2024-06-30 的随访两边都不要
- 2025 年数据：不用
- 重复提交：同一居民同一天只保留 created_at 最晚的一条（排序键 created_at、visit_id）
- 单位修正：身高 < 3 → ×100；体重与该居民此前随访体重中位数之比在 1.8～2.2 → ÷2；首次随访（此前没有体重）用 BMI > 40 判断，BMI 用修正后的身高算
- 哨兵与越界：收缩压 0 或 999 → 空；数值超出下面的范围 → 空（不删行）
    height_cm 120–210，weight_kg 30–200，sbp 60–260，dbp 30–150，exercise_min_week 0–3000，sleep_hours 2–14，waist_cm 50–150
- 类别写法：gender 只能是 M/F；smoking_status 只能是 never/former/current；drinking_freq 只能是 never/occasional/frequent；family_history_dm 只能是 yes/no/unknown（unknown 保留）
- 缺失：保留缺失，不填补、不删行（填补是模型的事）
- 年龄：随访日的周岁，整数：(visit_date − birth_date) 的天数 // 365.25，再转 int；应在 18–100 岁之间
- 输出列：19 个输入 + label + 6 个元数据 = 26 列（列名和顺序见要求 4）
- 输出文件：train.parquet、valid.parquet、summary.json
- 自检：visit_id 各自唯一且不跨集；训练最晚日期 <= 2024-03-02；验证全部在区间内；label 只有 0/1

【要求】
1. 严格按表实现。表里没写到的地方不要自作主张；必须做决定时，在代码注释里写"规格未定义，我选择了……"，并在最后单独列出所有这类决定。
2. 分成多个代码格，每格只做一件事，按这个顺序：导入 → 口径常量 → 读库与写法统一 → 去重 → 单位、哨兵、越界 → 标签函数 → 人群与时点切分 → 拼档案、算年龄、只留 26 列 → 落盘前自检 → 落盘与 summary.json。去重和单位修正在全部随访上做（包括已确诊者），之后才筛人群。
3. 所有口径数字（窗口天数、报告缓冲、快照日、验证区间、阳性阈值、取值范围、年龄范围）定义成常量，集中在第二格；代码里不出现裸数字。训练随访最晚日期由常量算出来，不要手写。
4. 列用显式白名单。输出列的名字和顺序固定为：
   age, height_cm, weight_kg, waist_cm, sbp, dbp, exercise_min_week, sleep_hours, veg_fruit_daily, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained, gender, smoking_status, drinking_freq, family_history_dm, label, visit_id, resident_id, visit_date, district, education, income_level
   known_diabetes、chronic_mgmt_tag 不得出现在输出里。两份表都按 visit_id 排序。
5. 写法统一之后，用 assert 检查每个类别列的取值都在规格表允许的集合里；漏了一种写法要报错停下。
6. 每一步清洗都打印"改了多少条"，0 条也要打印。
7. 缺失值保留，不填补、不删行；越界值置空，不删行。
8. 落盘前用 assert 检查：visit_id 各自唯一且两集无交集；训练最晚日期 <= 训练随访最晚日；验证全部在验证区间内；label 只有 0/1。
9. 输出 train.parquet、valid.parquet、summary.json，并把 summary.json 打印出来。
   summary.json 的结构固定为：
   {"source_db", "duplicates_removed", "weight_unit_fixed", "train_last_visit_allowed", "train": {"rows", "positives", "positive_rate", "date_min", "date_max", "residents"}, "valid": {同上}, "train_valid_resident_overlap"}
10. 不训练模型，不做独热编码，不画图。


## 8.1 创建新的文件：第0阶段之v2.ipynb

### 导入工具包

In [1]:
import pandas as pd
import numpy as np
import sqlite3                                              # 读 SQLite 数据库
import json                                                 # 写 summary.json、FROZEN.json
import os                                                   # 建目录、列文件
import shutil                                               # 复制文件（冻结时用）
import hashlib                                              # 算指纹（冻结时用）
import platform                                             # 记下 Python 版本
from sklearn.pipeline import Pipeline                       # 搭建流水线
from sklearn.compose import ColumnTransformer               # 三组列分开处理
from sklearn.impute import SimpleImputer                    # 填补缺失（在模型里做，不在数据集里做）
from sklearn.preprocessing import OneHotEncoder             # 对数据做独热编码
from sklearn.preprocessing import StandardScaler            # 对数据做标准化
from sklearn.linear_model import LogisticRegression         # 逻辑回归
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score   # 准确率、AUC、AP

### 口径常量：复盘的结论全部写在这里

In [2]:
DATA = "../data/"
DB = DATA + "incident.db"
OUT = "../output/v2/"                                       # 产物不再写到 Notebook 旁边
FROZEN = DATA + "frozen/"                                   # 冻结集：交给第一轮骨架的那一份（8.7–8.10）
TARGET = "label"

WINDOW_DAYS = 90                                            # 随访后多少天内的化验算数
MAX_REPORT_LAG_DAYS = 30                                    # 报告最晚比采样晚多少天（标签成熟缓冲）
TRAIN_END = pd.Timestamp("2024-06-30")                      # 训练快照日：只用这天已经知道的信息
VALID_START, VALID_END = pd.Timestamp("2024-07-01"), pd.Timestamp("2024-12-31")
DATA_AS_OF = pd.Timestamp("2025-06-30")                     # 原始库的导出日
FPG_POS, HBA1C_POS = 7.0, 6.5                               # 阳性阈值
TRAIN_LAST = TRAIN_END - pd.Timedelta(days=WINDOW_DAYS + MAX_REPORT_LAG_DAYS)   # 训练随访最晚的一天

NUM_COLS = ["age", "height_cm", "weight_kg", "waist_cm", "sbp", "dbp", "exercise_min_week", "sleep_hours"]
BIN_COLS = ["veg_fruit_daily", "hypertension_dx", "hyperlipidemia_dx", "gdm_history",
            "polyuria", "polydipsia", "weight_loss_unexplained"]
CAT_COLS = ["gender", "smoking_status", "drinking_freq", "family_history_dm"]
META_COLS = ["visit_id", "resident_id", "visit_date", "district", "education", "income_level"]
COLUMNS = NUM_COLS + BIN_COLS + CAT_COLS + [TARGET] + META_COLS                 # 26 列
ENUMS = {"gender": ["M", "F"], "smoking_status": ["never", "former", "current"],
         "drinking_freq": ["never", "occasional", "frequent"], "family_history_dm": ["yes", "no", "unknown"]}
RANGES = {"height_cm": (120, 210), "weight_kg": (30, 200), "sbp": (60, 260), "dbp": (30, 150),
          "exercise_min_week": (0, 3000), "sleep_hours": (2, 14), "waist_cm": (50, 150)}
AGE_RANGE = (18, 100)                                       # 筛查对象的年龄范围（成年人）
print("训练随访最晚:", TRAIN_LAST.date(), "  输出列数:", len(COLUMNS))

训练随访最晚: 2024-03-02   输出列数: 26


### 读库 + 类别写法统一

In [3]:
with sqlite3.connect(DB) as conn:
    residents = pd.read_sql("SELECT * FROM residents", conn, parse_dates=["birth_date"])
    visits = pd.read_sql("SELECT * FROM visits", conn, parse_dates=["visit_date", "created_at"])
    labs = pd.read_sql("SELECT * FROM lab_results", conn, parse_dates=["sample_date", "report_date"])

residents["gender"] = residents["gender"].replace({"male": "M", "男": "M", "female": "F", "女": "F"})
visits["smoking_status"] = (visits["smoking_status"].str.strip().str.lower()
                            .replace({"从不": "never", "吸烟": "current", "已戒": "former"}))
visits["drinking_freq"] = visits["drinking_freq"].replace({"不喝": "never", "偶尔": "occasional", "经常": "frequent"})

assert residents["gender"].isin(ENUMS["gender"]).all()                            # 写法统一了没有，让代码说
assert visits["smoking_status"].isin(ENUMS["smoking_status"]).all()
assert visits["drinking_freq"].isin(ENUMS["drinking_freq"]).all()
assert visits["family_history_dm"].isin(ENUMS["family_history_dm"]).all()
print(visits.shape)

(101297, 23)


### 重复提交：同一人同一天只留最后录入的一条

In [4]:
n0 = len(visits)
visits["_day"] = visits["visit_date"].dt.normalize()
visits = (visits.sort_values(["created_at", "visit_id"])                     # 排序键写全，谁是"最后一条"才唯一
          .drop_duplicates(["resident_id", "_day"], keep="last")
          .drop(columns="_day"))
n_dup = n0 - len(visits)
print("去掉重复提交:", n_dup, "条")

去掉重复提交: 437 条


### 单位、哨兵、越界：只用"当天及以前"的信息

In [5]:
visits = visits.sort_values(["resident_id", "visit_date"]).reset_index(drop=True)

n_height = (visits["height_cm"] < 3).sum()
visits.loc[visits["height_cm"] < 3, "height_cm"] *= 100                       # 按米录入
prior = (visits.groupby("resident_id")["weight_kg"]
         .transform(lambda s: s.shift().expanding().median()))                # 此前随访的体重中位数
bmi = visits["weight_kg"] / (visits["height_cm"] / 100) ** 2
jin = (visits["weight_kg"] / prior).between(1.8, 2.2) | (prior.isna() & (bmi > 40))  # 首次用 BMI 兜底
visits.loc[jin, "weight_kg"] /= 2                                             # 按斤录入

n_sentinel = visits["sbp"].isin([0, 999]).sum()
visits.loc[visits["sbp"].isin([0, 999]), "sbp"] = np.nan                      # 哨兵值
n_range = {}
for col, (lo, hi) in RANGES.items():                                          # 越界一律置空，不删行
    bad = visits[col].notna() & ~visits[col].between(lo, hi)
    n_range[col] = int(bad.sum())
    visits.loc[bad, col] = np.nan
print({"身高×100": int(n_height), "体重÷2": int(jin.sum()), "哨兵置空": int(n_sentinel)})
print("越界置空:", n_range)

{'身高×100': 610, '体重÷2': 520, '哨兵置空': 278}
越界置空: {'height_cm': 0, 'weight_kg': 0, 'sbp': 0, 'dbp': 0, 'exercise_min_week': 218, 'sleep_hours': 0, 'waist_cm': 31}


### 标签：截至 as_of 已出报告、随访后 90 天内的首次化验

In [6]:
def attach_label(v, as_of):
    """给随访挂标签。窗口内没有已出报告的化验 → 这条随访不返回（没化验 ≠ 阴性）。"""
    known = labs[labs["report_date"] <= as_of]                      # 站在 as_of 那天能看到的化验
    m = v[["visit_id", "resident_id", "visit_date"]].merge(known, on="resident_id")
    m = m[(m["sample_date"] >= m["visit_date"])
          & (m["sample_date"] <= m["visit_date"] + pd.Timedelta(days=WINDOW_DAYS))]
    first = m.sort_values(["sample_date", "lab_id"]).drop_duplicates("visit_id")  # 排序键写全，结果才唯一
    first[TARGET] = ((first["fpg_mmol_l"] >= FPG_POS) | (first["hba1c_pct"] >= HBA1C_POS)).astype(int)
    return v.merge(first[["visit_id", TARGET]], on="visit_id", how="inner")

### 人群 + 时点切分

In [7]:
screen = visits[visits["known_diabetes"] == 0]                                   # 已确诊者不是筛查对象
train = attach_label(screen[screen["visit_date"] <= TRAIN_LAST], as_of=TRAIN_END)
valid = attach_label(screen[screen["visit_date"].between(VALID_START, VALID_END)], as_of=DATA_AS_OF)
print("训练", len(train), "  验证", len(valid))

训练 32632   验证 13067


### 拼档案、算年龄、只留 26 列

In [8]:
def finalize(d):
    d = d.merge(residents, on="resident_id", how="left")
    d["age"] = ((d["visit_date"] - d["birth_date"]).dt.days // 365.25).astype(int)   # 随访日的周岁
    return d[COLUMNS].sort_values("visit_id").reset_index(drop=True)

train, valid = finalize(train), finalize(valid)
print(train.shape, valid.shape)

(32632, 26) (13067, 26)


### 落盘前自检：写错了就在这里炸

In [9]:
assert train["visit_id"].is_unique and valid["visit_id"].is_unique
assert not set(train["visit_id"]) & set(valid["visit_id"]), "同一次随访跨集"
assert train["visit_date"].max() <= TRAIN_LAST, "训练集含标签未成熟的随访"
assert valid["visit_date"].between(VALID_START, VALID_END).all(), "验证集超出区间"
assert train[TARGET].isin([0, 1]).all() and valid[TARGET].isin([0, 1]).all()
assert "chronic_mgmt_tag" not in COLUMNS and "known_diabetes" not in COLUMNS
print("五条自检全部通过")

五条自检全部通过


### 落盘 + 数据说明 summary.json

In [10]:
os.makedirs(OUT, exist_ok=True)
train.to_parquet(OUT + "train.parquet", index=False)
valid.to_parquet(OUT + "valid.parquet", index=False)

def describe(d):
    return {"rows": len(d), "positives": int(d[TARGET].sum()),
            "positive_rate": round(float(d[TARGET].mean()), 4),
            "date_min": str(d["visit_date"].min().date()), "date_max": str(d["visit_date"].max().date()),
            "residents": int(d["resident_id"].nunique())}
summary = {"source_db": "incident.db", "duplicates_removed": n_dup, "weight_unit_fixed": int(jin.sum()),
           "train_last_visit_allowed": str(TRAIN_LAST.date()),
           "train": describe(train), "valid": describe(valid),
           "train_valid_resident_overlap": len(set(train["resident_id"]) & set(valid["resident_id"]))}
json.dump(summary, open(OUT + "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "source_db": "incident.db",
  "duplicates_removed": 437,
  "weight_unit_fixed": 520,
  "train_last_visit_allowed": "2024-03-02",
  "train": {
    "rows": 32632,
    "positives": 5285,
    "positive_rate": 0.162,
    "date_min": "2023-01-01",
    "date_max": "2024-03-02",
    "residents": 26565
  },
  "valid": {
    "rows": 13067,
    "positives": 1675,
    "positive_rate": 0.1282,
    "date_min": "2024-07-01",
    "date_max": "2024-12-31",
    "residents": 12806
  },
  "train_valid_resident_overlap": 10633
}


## 8.2 验收清单

### 在 v2 上训练同一个模型

In [11]:
X_train, y_train = train[NUM_COLS + BIN_COLS + CAT_COLS], train[TARGET]
X_valid, y_valid = valid[NUM_COLS + BIN_COLS + CAT_COLS], valid[TARGET]
encoder = OneHotEncoder(handle_unknown="ignore")
pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),  # 缺失也是信息
                          ("scale", StandardScaler())]), NUM_COLS),
        ("bin", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), BIN_COLS),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", encoder)]), CAT_COLS),
    ])),
    ("lr", LogisticRegression(max_iter=2000)),
])
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_valid)[:, 1]
k = int(round(len(valid) * 0.15))                                 # 15% 名额
hit = y_valid.to_numpy()[np.argsort(-proba)[:k]].sum()
print({"患病率": round(float(y_valid.mean()), 3),
       "准确率@0.5": round(accuracy_score(y_valid, proba >= 0.5), 3),
       "ROC-AUC": round(roc_auc_score(y_valid, proba), 4),
       "AP": round(average_precision_score(y_valid, proba), 4),
       "R@15": round(float(hit / y_valid.sum()), 3), "P@15": round(float(hit / k), 3)})

{'患病率': 0.128, '准确率@0.5': 0.929, 'ROC-AUC': 0.9329, 'AP': 0.7665, 'R@15': 0.747, 'P@15': 0.639}


## 8.3 冻结

验收清单回答的是"口径对不对"，所以允许 ±1% 的出入。接下来要回答的是另一个问题：后面所有的工作，用的是不是同一份数据。第一轮要照着文档一步步复现，文档里写"训练 32,632 条、阳性 5,285"，每个人的 manifest 都得是这个数；第二轮要把 v2 搬进框架、从 incident.db 再重建一遍，重建的结果要和今天这份逐格比对。两件事都需要一份定下来、不再改的数据。把它定下来，叫冻结。

### 冻结前：按下游读数据时会查的，再查一遍

In [12]:
def check_frozen(d):
    """冻结前的入口检查：列、编号、标签、写法、取值范围。有问题一次列全，不在第一个问题就停。"""
    if list(d.columns) != COLUMNS:
        return [f"列不对：多了 {sorted(set(d.columns) - set(COLUMNS))}，少了 {sorted(set(COLUMNS) - set(d.columns))}"]
    problems = []
    if d["visit_id"].isna().any() or not d["visit_id"].is_unique:
        problems.append("visit_id 有缺失或重复")
    if not d[TARGET].isin([0, 1]).all():
        problems.append("label 不全是 0/1")
    if not pd.api.types.is_datetime64_any_dtype(d["visit_date"]):
        problems.append("visit_date 不是日期类型")
    for col, allowed in ENUMS.items():                                   # 类别：只能是规定的写法，不能缺
        n = int((~d[col].isin(allowed)).sum())
        if n:
            problems.append(f"{col}: {n} 行不在 {allowed} 里")
    for col in BIN_COLS:                                                 # 二元：只能是 0/1，允许缺
        n = int((d[col].notna() & ~d[col].isin([0, 1])).sum())
        if n:
            problems.append(f"{col}: {n} 行不是 0/1")
    for col, (lo, hi) in {**RANGES, "age": AGE_RANGE}.items():           # 数值：在范围内，允许缺
        n = int((d[col].notna() & ~d[col].between(lo, hi)).sum())
        if n:
            problems.append(f"{col}: {n} 行超出 [{lo}, {hi}]")
    return problems

train = pd.read_parquet(OUT + "train.parquet")                           # 查的是验收过的那两份文件
valid = pd.read_parquet(OUT + "valid.parquet")
print("train 的问题:", check_frozen(train))
print("valid 的问题:", check_frozen(valid))
missing = pd.DataFrame({"train": train.isna().mean(), "valid": valid.isna().mean()}).round(3)
print(missing[(missing["train"] > 0) | (missing["valid"] > 0)])          # 只看有缺失的列

train 的问题: []
valid 的问题: []
                         train  valid
waist_cm                 0.152  0.151
sbp                      0.003  0.003
exercise_min_week        0.002  0.002
sleep_hours              0.080  0.081
gdm_history              0.480  0.478
polyuria                 0.030  0.027
polydipsia               0.029  0.030
weight_loss_unexplained  0.030  0.031
income_level             0.171  0.168


### 两种指纹：内容指纹、文件指纹

In [13]:
def content_sha256(d):
    """内容指纹：固定列序、数值统一成浮点、按 visit_id 排好，写成 CSV 文本再算 sha256。
    只认内容：文件格式、库版本、行的顺序、存成整数还是浮点，都不影响它。"""
    canon = d[COLUMNS].astype({c: "float64" for c in NUM_COLS + BIN_COLS + [TARGET]})
    text = canon.sort_values("visit_id").to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def file_sha256(path):
    """文件指纹：逐块读文件算 sha256。用来记住原始库是哪一份。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

print("train 内容指纹:", content_sha256(train)[:16])
print("valid 内容指纹:", content_sha256(valid)[:16])
print("incident.db 文件指纹:", file_sha256(DB)[:16])
print("打乱行序后:", content_sha256(train.sample(frac=1, random_state=0))[:16])     # 行序不影响内容指纹

train 内容指纹: 52e41f28ec7da2b0
valid 内容指纹: b489f7c685ee85ec
incident.db 文件指纹: 412b7c1adb612f9f
打乱行序后: 52e41f28ec7da2b0


### 写出冻结集：两份数据 + 一张身份证 FROZEN.json

着重看 身份信息文件：FROZEN.json，这里面就是 口径。

In [14]:
card = {
    "name": "stage0-v2",
    "built_by": "第 0 阶段 v2 Notebook 单元 38–47（口径见规格表 8.1）",
    "source": {"file": "incident.db", "sha256": file_sha256(DB)},
    "spec": {"population": "known_diabetes == 0", "window_days": WINDOW_DAYS,
             "max_report_lag_days": MAX_REPORT_LAG_DAYS, "fpg_pos": FPG_POS, "hba1c_pos": HBA1C_POS,
             "train_end": str(TRAIN_END.date()), "train_last_visit": str(TRAIN_LAST.date()),
             "valid_start": str(VALID_START.date()), "valid_end": str(VALID_END.date()),
             "data_as_of": str(DATA_AS_OF.date())},
    "columns": {"numeric": NUM_COLS, "binary": BIN_COLS, "categorical": CAT_COLS,
                "label": TARGET, "meta": META_COLS},
    "cleaning": {"duplicates_removed": n_dup, "height_m_to_cm": int(n_height), "weight_jin_to_kg": int(jin.sum()),
                 "sbp_sentinel_to_null": int(n_sentinel), "out_of_range_to_null": n_range},
    "train": {"file": "train.parquet", **describe(train), "content_sha256": content_sha256(train)},
    "valid": {"file": "valid.parquet", **describe(valid), "content_sha256": content_sha256(valid)},
    "train_valid_resident_overlap": len(set(train["resident_id"]) & set(valid["resident_id"])),
    "env": {"python": platform.python_version(), "pandas": pd.__version__},
}
if os.path.exists(FROZEN + "FROZEN.json"):                                # 冻结过：只核对，不覆盖
    old = json.load(open(FROZEN + "FROZEN.json", encoding="utf-8"))
    same = all(old[s]["content_sha256"] == card[s]["content_sha256"] for s in ["train", "valid"])
    assert same, "data/frozen/ 里已经有一份内容不同的冻结集：不要覆盖，先查是哪条口径变了"
    print("已经冻结过，内容一致，不重复写")
else:
    os.makedirs(FROZEN)
    for s in ["train", "valid"]:
        shutil.copyfile(OUT + s + ".parquet", FROZEN + s + ".parquet")     # 冻结的就是验收过的那两份文件
    json.dump(card, open(FROZEN + "FROZEN.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("已冻结到", FROZEN)
print(sorted(os.listdir(FROZEN)))

已经冻结过，内容一致，不重复写
['FROZEN.json', 'train.parquet', 'valid.parquet']


| 段 | 回答什么 |
|---|---|
| `source` | 从哪个库来 |
| `spec` | 按什么口径做的：人群、窗口、阈值、快照日、验证区间 |
| `columns` | 哪些是模型输入，哪些是元数据 |
| `cleaning` | 每条清洗改了多少条 |
| `train` / `valid` | 多少行、多少阳性、什么时间段、内容指纹 |
| `env` | 用什么环境做的 |

### 读回来核对：冻结集只认指纹

In [15]:
card = json.load(open(FROZEN + "FROZEN.json", encoding="utf-8"))
for s in ["train", "valid"]:
    d = pd.read_parquet(FROZEN + card[s]["file"])
    assert content_sha256(d) == card[s]["content_sha256"], f"{s}: 读回来的内容和冻结时不一样"
    assert len(d) == card[s]["rows"] and int(d[TARGET].sum()) == card[s]["positives"]
    assert not check_frozen(d), f"{s}: 入口检查没过"
    print(s, card[s]["rows"], card[s]["positives"], card[s]["content_sha256"][:16])
print("来源", card["source"]["file"], card["source"]["sha256"][:16])

train 32632 5285 52e41f28ec7da2b0
valid 13067 1675 b489f7c685ee85ec
来源 incident.db 412b7c1adb612f9f


## 9. 从问题到架构

### 1. v2 已经写对了，为什么还不够？

v2 数据集、验收、冻结，我们全部做完了，**数据本身是对的**，但它只是一份 "**在笔记本里跑通的脚本**"。它只能保证 "今天这份数据没问题"，管不住以后、管不住别人、管不住上线之后出的各种问题。这些问题，靠再多写几行 pandas 代码解决不了，要靠工程架构来约束。

例如：

| 明天可能发生的事 | 通俗解读 | v2 为什么拦不住 |
|---|---|---|
| 同事新开一个 Notebook 重做数据，口径变回默认值 | 你写的筛选、清洗规则只藏在你这个 Notebook 里面。别的同事新建文件，很容易忘记全部筛选条件，跑出一套不一样的数据 | 冻结只锁死了今天产出的这一份 parquet 文件，没有把【口径规则】变成所有人必须共用、不能随便改的标准组件 |
| 有人偷偷改参数：把 TRAIN_LAST 时间改掉、把确诊字段 known_diabetes 加回来 | 有人改了时间边界、把泄露字段放回去，代码依然能正常跑完，不会自动报错 | 校验断言只写在当前这个脚本里；换一个脚本跑，这些约束就消失了，没有统一的强制校验骨架 |
| 线上服务还在用旧的 predict.py | 训练的时候一套预处理代码，上线推理是另一套代码，两边逻辑不一样，就是前面讲过的线上线下不一致 | v2 只负责产出训练用的数据，没有把预处理、模型打包成统一、可复用的推理流水线，训练和服务是分开两套东西 |
| 过几个月问：这个模型是用哪一份数据、哪个版本代码训练出来的？ | FROZEN.json 只记录了数据集是谁；但是模型、训练脚本、当时的参数没有绑定存档，查不到完整链路 | 只有数据集身份证，没有模型实验的身份证，数据和训练过程是脱节的 |
| 换一台电脑、重装依赖再跑，结果变了 | 你的代码里写死了本地路径，依赖包版本没锁死，换环境就翻车 | 没有锁定环境、统一路径配置，代码是 "跟着你这台电脑才能跑"，不是到处都能稳定复现 |
| 上线半年，人群患病率从 16.2% 漂移了，我们完全不知道 | 人群慢慢变了，模型效果慢慢下滑，但是没人发现 | v2 只管做完训练，没有上线预测日志、没有漂移监控，上线之后就是黑盒 |

**区分「脚本」和「架构」**

（1）脚本（我们现在写的 v2、清洗、冻结代码）：**一次性把一份数据跑对**，它是一次性的操作。它能回答：我这次生成的数据，对不对？

（2）架构（后面要学的骨架）：**建立一套规则体系，约束所有人、所有时间、所有环节。**

它回答：代码放在哪里、谁可以调用、谁来自动检查、每一步怎么绑定存档、上线之后怎么监控

例子：

> 脚本 = 你亲手做了一份合格的试卷，并且复印锁进档案盒（就是冻结）。
>
> 架构 = 一套标准化的出题系统：题库、固定判分规则、权限、日志、自动校验。
>
> 只锁好一份试卷，拦不住后面有人重新出题、偷偷改题目、判卷标准变了

**v2 做对 ≠ 整个项目稳了**；v2 解决的是单份数据集的正确性

验收、冻结，只是解决 "这一份数据" 的校验和存档；它约束不了后续的人、后续的实验、线上服务.

### 2. 解决方案

解决方案推导表：先看到风险问题 → **提炼出一条工程原则** → 再落地成具体的技术机制

逻辑链条：问题（K/E）→ 我们要遵守什么原则 → 用什么工具 / 代码组件实现这个原则

例如：

（1）原则：哪些列能进模型，必须显式写下来，机制：特征目录、输入白名单、元数据、禁用列

> 例如：不能谁写代码随手挑列。要提前定好一张清单，哪些特征允许用、哪些是泄露字段禁止用。不是靠人记，有白名单，不在清单里的列直接不让进模型

（2）原则：环境、路径、口径常量都要声明｜机制：uv.lock、仓库骨架、配置系统

> 例如：之前踩坑：换电脑、换环境就跑不出一样结果。所以包版本、文件路径、时间阈值这些固定参数，不能散写在 Notebook 里，统一放到配置文件，uv.lock 锁定依赖版本，保证在哪跑环境都一样。

（3）原则：规则由机器守，不由人守｜机制：质量门、架构测试、AGENTS.md

> 例如：人是最容易忘、容易改错的。筛选规则、校验逻辑不能靠 "自觉"。写成自动化测试，每次提交代码自动跑；不满足规则，直接卡住不让提交，机器把关，而不是人工审核

（4）原则：数据集与模型输入分开｜机制：契约①：26 列宽表 + 列检查

> 例如：原始冻结数据集里面字段很多，有元数据、ID、标签；模型只取约定好的 26 列。这就是数据契约：数据集是原材料，模型输入是约定好的接口，列不对、顺序不对直接报错，防止偷偷加字段

根据上面问题，生成了三刀切的方式进行设计。

### 3. 第一刀：按模型的一生，切成六个阶段

第一刀划分：按照模型完整生命周期，把前面所有 K、E 风险，分到它实际发生的环节

图 9-1 A 是地基，B 到 F 首尾相接成环；F 的漂移报告和线上效果让人决定是否重训，重训又从 B 开始

整个生命周期一共 6 块：A 是地基（底层设施），B～F 是模型流转的闭环，F 跑完还能回到 B，形成循环，也就是模型持续迭代

（1）A 底层设施（地基，所有阶段都依赖）

要回答的问题：怎么保证所有人，都按同一套规则写代码？

交付下游：分层契约、注册表、配置、特征目录

例如：这是整个项目的 "公共规矩底座"。统一的仓库骨架、依赖锁定、特征白名单、自动化质量门禁。不管是谁，不管是做数据、训模型、上线服务，都要遵守这套统一规则。它不是单独的业务流程，是所有环节共用的基础，所以说了：A 不算业务流程里的一层，一半是代码包（特征目录、注册表），一半是提交时的自动化检查（质量门、架构测试）

前面遇到的：换电脑环境跑不一样、同事新开 Notebook 口径乱改、随便加特征，这些坑，根源都在 A 这里

（2）B 数据准备

要回答的问题：训练数据从哪来？干净吗？标签对吗？有没有偷偷用未来的数据（时间泄露）？

交付下游：数据集 + manifest（也就是我们写的 FROZEN.json）

例如：就是我们前面一大段一直在做的工作：拉原始数据、清洗、去重、构建标签、时间切分、验收、冻结、生成数据身份证。

这个环节产出**冻结好的可信数据集**，交给后面训练环节。

泄露、标签错误、清洗计数不对、切分规则混乱，全部发生在 B 阶段

（3）C 训练与评估

要回答的问题：选什么模型？模型效果好不好？指标达到上线标准了吗？指标能不能追溯？

交付下游：训练运行记录

例如：拿到冻结数据集之后，跑 Pipeline 训练模型、算 AUC、AP、R@15% 这些指标，用 MLflow 记录实验。这里重点不是 "跑出一个高分"，而是指标要带口径、有记录、有上线门槛；同时预处理和模型打包在一起（Pipeline），为后面打包 ModelBundle 做准备。

风险点：指标造假、没有记录、预处理和模型分开写，都属于 C 阶段。

（4）D 发布与注册

要回答的问题：这个模型能不能上线？谁审批？出问题能不能回滚到老版本？

交付下游：ModelBundle（打包：预处理 + 模型 + 阈值，一个整体）

例如：训练好的模型不能直接丢上线。要做发布评审、版本登记、打包成 ModelBundle。这个 Bundle 是一个完整包裹，预处理、模型、判定阈值全部封在里面，后面服务直接加载这个包，避免训练和推理两套代码。

线上线下不一致的源头，很多时候就是发布环节没有统一打包。

（5）E 服务与观测

要回答的问题：怎么把模型变成可以调用的稳定服务？服务跑起来正常吗？

交付下游：OpenAPI 接口 + 预测记录

例如：把 ModelBundle 部署成接口，前端可以调用它做预测；每一次预测请求，都要保存预测日志。**重点：服务只加载打包好的 Bundle，不再手写一套预处理代码，保证训练、推理逻辑一模一样。**这里产出的预测记录，是后面监控环节的原料。

（6）F 监控与回流

要回答的问题：上线之后人群分布变没变？模型效果下滑没有？什么时候要重新训练？

交付下游：漂移报告，触发之后回到 B（重新准备数据，重训模型）

例如：模型上线不是结束，只是开始。

持续看线上特征漂移、等真实标签回流，评估模型效果。一旦漂移严重、效果掉到阈值以下，就生成漂移报告，回到 B 环节，重新整理数据、重新训练，形成闭环。

划分的目的：**定位故障在哪一段。**以后出问题，先判断：是数据阶段 B 炸了？还是训练 C？还是发布 D？还是线上监控 F？

### 4. 第二刀：阶段之间只通过契约交互

才第一刀，把模型生命周期切成了 A、B、C、D、E、F 六个阶段。

第二刀，就是规定清楚：阶段和阶段之间，只能靠【契约】来交接，不能随便传一堆零散文件、裸 DataFrame。

就是上下游交接没有标准，B 给 C、D 给 E 的时候，东西长啥样没有约定，很容易两边理解不一样，出现 "落盘不是预期数据集"、"训练和推理两套代码"

讲 5 份契约：

（1）数据集 + manifest｜B → C（数据准备交给训练评估）

里面包含：固定 26 列宽表、来源指纹、行数、阳性样本数量、时间边界、配置指纹。

例如：B 环节交付给训练环节的不是随便一个 csv，是**固定格式的数据集 + 它的身份证清单**。C 拿到手第一件事就对照 manifest 校验，列不对、行数不对、指纹不对，直接拒绝运行。

（2）训练运行｜C → D（训练评估交给发布注册）

里面包含：整条 Pipeline、带口径的评估报告、决策策略（分数→阈值）。

例如：练完不能只丢一个 model 文件。要把整套流水线、本次评估指标、用多大阈值划分高危人群一起打包交给发布环节。指标绑定数据集口径，方便追溯，不能只有一个孤零零 AUC 数字。

（3）ModelBundle｜D → E（发布注册交给线上服务）

里面包含：Pipeline、元数据、输入列清单、特征目录指纹、阈值、版本信息。

例如：这就是解决线上线下不一致的核心包。预处理 + 模型 + 判定阈值全部捆在一起。服务端只加载这个 Bundle，不再自己重写一遍清洗、编码代码，训练和推理走完全一样的逻辑

（4）OpenAPI｜E → 前端（服务交给前端页面）

里面包含：请求字段、中文名、取值范围、分类枚举，由特征目录自动生成。

例如：前端调用模型服务，接口长什么样、每个参数填什么、哪些值合法，全部写死在契约里。前端不用猜，照着契约写页面；接口变了契约先改。

（5）预测记录 v1｜E → F（线上服务交给监控回流）

里面包含：每次预测一条记录，请求号、随访编号、模型版本、预测分数、风险等级、用到的特征、schema 版本号。

例如：监控环节没有数据就没法做漂移检测。线上每一次预测都按统一格式落日志，这一份日志就是 F 阶段监控、标签回流的原材料。

#### 4.1 有契约之后，带来三个很大的好处

（1）阶段内部可以随便重构，只要契约不变，下游不受影响

（2）可以分工并行开发

（3）换新项目，整套骨架不用推翻，只改契约里面的字段

### 5. 第三刀：三个包，依赖只能向下

前面第一刀切生命周期、第二刀定阶段之间的交接契约，第三刀是做代码分包，定死依赖规则：依赖只能向下，不能横向、不能向上依赖。

之前的老问题：训练脚本、预测脚本各自写一套特征处理代码，两边逻辑不一样，造成线上线下不一致。核心思路：共用的代码抽到底层核心包，训练包、服务包都去依赖核心包，但是训练包和服务包互相不能依赖

图 9-2 三个 Python 包加一个前端；服务和训练都依赖核心包，彼此互不依赖。

#### 5.1四个模块的解读

（1）drisk-core 核心包（底层，最基础）

导入名：diabetes_risk

存放内容：训练和线上推理两边都必须一模一样的代码。注册表基类、特征目录、特征工程清洗逻辑、决策阈值规则、ModelBundle 打包解析、日志工具。

允许依赖：**无，它是最底层。**

通俗理解：这是唯一一份 "特征处理真值代码"。训练的时候调用它做特征变换，上线推理也调用这同一份代码，不会再写两套

（2）drisk-training 训练包

导入名：drisk_training

存放内容：只属于训练环节的东西：数据读取、数据集切分、模型训练、评估画图、MLflow 追踪、流水线脚本、训练命令行。

允许依赖：只能依赖 core。

通俗理解：专门用来训模型的包。里面可以装 Optuna、绘图、DVC 这些训练才需要的重型依赖。**它不负责对外提供预测接口。**

（3）drisk-api 服务包

导入名：drisk_api

存放内容：FastAPI 推理接口、请求接收、返回结果、鉴权。

允许依赖：只能依赖 core。

通俗理解：线上跑的推理服务。它只加载已经打包好的 ModelBundle，调用 core 里面的特征逻辑，**里面不能出现训练代码、画图库、调参工具。**线上镜像瘦身、安全干净。

（4）admin-web 前端

存放内容：后台页面、风险展示、复查列表。

允许依赖：**不直接 import 任何 Python 包，只通过 HTTP 接口调用 api 服务。**

通俗理解：前端和后端 Python 代码完全隔离，只认 OpenAPI 契约。

总结三刀的内容：

第一刀（按模型一生切 6 阶段）：知道一件事属于哪个环节，定位问题；

第二刀（阶段之间靠契约交互）：上下游交接标准固定，解耦、可以并行开发；

第三刀（代码分包，依赖只能向下）：把共用逻辑下沉，从代码层面保证训练、推理共用同一套特征代码，根治线上线下不一致。

### 6. 变化点收口：注册表、配置、特征目录

前面我们做好了生命周期、契约、分包。这一节解决另一个大痛点：**项目里的可变参数、模型组件、字段定义散落在代码各处，改一个地方，到处都要手动改，很容易漏改、不一致。**

我们把三类变化点，收拢到三个专门机制统一管理：**组件注册表、配置、特征目录**，做到同类信息只写一处，一处修改，全局生效。

#### 6.1 回顾 V1版本的老问题：

1. 可替换组件：比如逻辑回归、随机森林两个模型，代码里写一堆 if 判断选哪个；
2. 口径硬编码数字：90 天随访窗口、7.0 的血糖阈值、0.3 的分数门槛、150 这类数值，直接写在代码行里；
3. 字段信息：哪些特征、类型、取值范围、中文名称，训练、校验、前端接口各自写一遍。散落在各处，改阈值要搜全工程，新增模型要加 if 分支，新增字段要在训练、校验、前端三处同步，非常容易出错

推出三种解决方式：

- 组件注册表：同类、可替换、会增长的东西

    - Registry 是一个通用注册容器，专门用来装同一类、可以互相替换的东西，这里是模型；
    - register 是装饰器，给模型起名字，登记进注册表；新增模型的时候，只需要加装饰器，**不用写 if-else 分支**；

- 配置：口径数字的唯一来源（yaml）

    **所有业务口径数字**，比如随访窗口 90 天、血糖阳性阈值 7.0、数据集划分时间，**全部从代码里抽出来放到 YAML 配置，代码不再写硬编码数字。**

    并且支持配置继承：base.yaml 放通用默认参数；不同实验的 yaml 只写和 base 不一样的参数，不用重复复制一大段。

- 特征目录：字段信息只写一次

    **所有特征的定义，只在这里写唯一一份**：字段英文名、类型（数值 / 二分类 / 多分类）、合法取值范围、前端展示中文名称

    同时标记 forbidden 禁止使用字段，防止把 ID、标签、泄露特征不小心放进模型；derived 标记衍生特征（BMI 这种由身高体重算出来的）

#### 6.2 串联前面三刀，形成完整闭环

第一刀：划分生命周期 B/C/D/E/F；

第二刀：阶段之间靠契约交接；

第三刀：代码分包，依赖单向向下；

这一节：**把系统里面所有经常变的东西，收拢到三个中心化的地方管理**，不让魔法数字、字段定义、组件选择散落在代码各个角落。这是工程落地里面，保证长期可维护、可复现、不容易口径跑偏的关键一步。

**变化点中心化收口。**

凡是会经常修改、多地方要用到的信息，不要散写在业务代码里，单独抽成配置 / 注册表 / 特征目录，一处修改，全局统一，减少不一致 bug

### 7. 规则交给机器守：质量门

前面我们定了一大堆规范、注册表、配置、特征目录、分包契约，但是有一个很大隐患：**人也好、AI 也好，写代码的时候偷偷把规则改了，没人发现怎么办？**

规则写下来以后，谁去检查它没被改回去？

靠人工盯是不靠谱的，人会忘、会偷懒，AI 为了跑通代码，还会悄悄改口径、改基线、绕过约束，而且不留痕迹。这一节的思路就是：**把检查规则交给机器自动看守，也就是【质量门】。**

#### 7.1什么是质量门

**质量门就是一道关卡，必须全部检查项通过，才算合格；只要有一项失败，直接卡住，不能往下走。**

整个项目只靠一条命令触发：

```
uv run python scripts/ci.py
```

这条命令内部会按顺序自动跑一整套检查：

1. ruff 代码规范检查（代码格式、代码坏味道）
2. ruff 自动格式化
3. mypy 静态类型检查，防止类型乱传
4. import-linter：检查分层依赖契约，不允许跨层乱依赖
5. pytest 单元测试 + 代码覆盖率校验

**任何一步失败，整体直接判定失败。**这是项目唯一合格标准，不依赖 GitHub 那种线上 CI，本地也能跑。

好处：不管是你自己写代码，还是 AI 帮你改代码，提交之前，本地先跑这条命令，机器先把关。

#### 7.2 重点：架构测试，专门防 AI "耍小聪明绕规则"

最关键的点，讲清楚 AI 的通病：AI 有时候为了让测试跑绿、让代码不报错，会偷偷破坏我们之前定下的架构规则，相当于 "改答案凑结果"。所以我们单独加了一组架构测试，卡死这些绕过手段：

- **冻结集、冻结基线的指纹不能变**：基线就是我们固定好的标准答案数据集，指纹相当于它的唯一身份证。不能为了适配代码，偷偷换数据集、改标签，不能改答案迁就代码。
- **禁止 `sys.path.insert`、`os.chdir`**：这是早期代码很常见的坏毛病，强行改 Python 导入路径，会破坏我们前面设计的单向分包依赖，环境一变代码就崩。
- **质量门槛（覆盖率等）不能调低、分层契约不能删减**：不能测试跑不通，就手动把覆盖率要求从 80% 改成 30%，不能删掉依赖约束。
- **服务包不能依赖训练包、pytest 配置不能随便加 pythonpath**：守住分包单向依赖，防止训练代码泄漏到推理服务里。

例如：

> 我们定好考试规则：不能改试卷标准答案、不能抄近路、不能降低及格线。

质量门的架构测试，就是监考老师，专门抓这种偷偷改规则、走后门的行为。

#### 7.3 和前面知识点串联

1. 划分生命周期、切分阶段；
2. 阶段之间用契约对接；
3. 代码分包，单向依赖；
4. 变化点收口：注册表、配置、特征目录，把可变的东西收拢；

到这一步：**质量门 + 架构测试，用来保护上面所有规则不被偷偷破坏。**

前面是 "我们要做成什么样"，质量门是 "机器守住底线，不许偷偷改回去"。

总结：

质量门 = 自动化守门人，把架构、数据、代码规范变成机器可执行的检查；

不依靠人工审核，本地一条命令就能校验，专门防范人工疏忽、AI 悄悄改口径、绕开架构约束

### 8. v2 的每一段，搬到哪里

搭框架不是推翻 v2 重写，而是把 v2 的每一段搬进它该在的位置。

搭新版框架，不是把原来 v2 写好的代码全部扔掉重写，而是做 "搬家归类"：把原来 v2 里每一段代码，放到它最合适的文件夹 / 模块里，搬完之后，再给它加上自动检查、自动校验，也就是上一节咱们讲的质量门。

例如：

大家可以把 v2 想象成一个乱糟糟的工具箱，锤子、螺丝刀、尺子、说明书、零件全都混在一个大箱子里，要用的时候随手拿，写死在代码里，改一处漏一处，出问题很难排查。

现在搭建规范框架，不是扔掉所有工具，而是：

1. 给工具箱分格子、分抽屉；
2. 每一样工具放到固定抽屉；
3. 放好之后，给每个抽屉加一把检查锁（自动校验），放错、参数乱改、数据不对，机器直接报警。

| v2 的哪一格 | 搬到哪里 | 搬过去之后多了什么 |
|---|---|---|
| 单元 39 口径常量 | `configs/base.yaml` 的 `label` / `split` / `dataset` 段 | 类型校验；改一个数，所有用到的地方同时变 |
| 单元 39 四组列名、`ENUMS`、`RANGES` | `configs/features.yaml` | 范围、缺失上限、中文名、禁用列 |
| 单元 40–42 规范写法、去重、单位、哨兵 | 清洗规则（每条一个注册项） | 每条输出"改了多少条"；配单元测试 |
| 单元 43 标签函数 | 标签模块 | 窗口、阈值来自配置；边界有单元测试 |
| 单元 44 人群 + 时点切分 | 切分断言 + 严格时点构建器（`strict_pit`） | 任何数据来源都要过同一关 |
| 单元 45 只留 26 列 | 契约①的列检查 | 缺列一次列全，不是一句 `KeyError` |
| 单元 46 落盘前自检 | 切分完整性检查 | 失败时给出条数和日期，而不是一句 `AssertionError` |
| 单元 47 `summary.json` | `manifest.json` | 文件指纹、配置指纹、目录指纹、git 版本 |
| 单元 49–52 冻结 | `data/frozen/` 原样进仓库，由 `frozen` 构建器读（B-L1） | manifest 的行数、阳性数对 `FROZEN.json`；第 2 轮 `strict_pit` 的重建结果对内容指纹（对账点 1） |
| 单元 48 训练评估 | ModelSpec、Pipeline、评估报告、MLflow | 预处理随模型保存；每次训练是一条可追溯记录 |
| 上午的 `model.pkl` + `predict.py` | ModelBundle + FastAPI 服务 | 一个包、一条代码路径；一致性测试 |

前面我们学：划分生命周期、单向依赖分包、质量门自动检查。

这一节就是落地动作：

- 不是推翻老代码，是存量代码迁移重构
- 每一段逻辑归位到固定位置（配置、清洗、标签、切分、模型、服务）
- 每搬完一块，就补上校验、日志、单元测试，交给质量门自动看守

搬家重构 = 逻辑归位 + 契约约束 + 自动校验；老代码是原料，框架是收纳规则，质量门是看门的。

## 10. 架构要先行，先建哪一块？

### 1. 架构先行，不等于一次建完

很多同学会误解：架构先行 = 一口气把所有模块、所有代码全部一次性写完，再开始跑模型、做测试。

不对！**架构先行 ≠ 一次性全部建好。**这里给大家对比两种开发思路，重点理解【迭代增量，行走的骨架】

例子：搭房子

- **做法一：水平分层（传统逐层做完）**

想象盖房子：先把地基全部做完，再把一楼整层全部完工，再二楼全部完工，一直到顶楼。等到最后，才把水电管线全部接通、第一次通电测试。

隐患：如果一开始水电接口设计错了，要等到全部盖得差不多了才发现，前面墙、楼板都已经砌好了，改起来代价巨大。

顺序：A 全部做完 → B 全部做完 → C 全部做完 → D 全部做完 → E 全部做完 → F 全部做完

这个项目一共 30 个步骤，要等到第 22 步，才能第一次调用预测功能，到 24 步整个流程才闭环。

**它最大的问题不是慢，而是问题暴露太晚：**

举咱们机器学习项目的例子：一开始定义 ModelBundle 模型包的时候，少约定了一个字段；或者预测记录表少设计了一列。这个错误，在前面 A、B、C 阶段根本发现不了，一直写到 E、F 阶段才爆出来。

这时候前面已经写了大量代码，接口和契约写错了，改起来牵一发而动全身，很多代码都要返工。

- **做法二：迭代增量（行走的骨架）**

先搭一个极简骨架：先打地基，然后柱子、简易楼板、最简水电先全部接通，房子虽然简陋，但是**能通电、能走通完整流程**。

第一轮：只是个毛坯架子，五脏俱全，能跑通完整闭环；

第二轮：再把墙体加厚、做粉刷；

第三轮：再增加家具、扩展功能。

好处：随时停下来，它都是一个完整、能跑的房子；接口、契约在很早的时候就能验证对错。

轮次拆分：

- **第 0 轮**：只做 A 地基（基础目录、基础规则、基础配置）
- **第 1 轮 L1**：B、C、D、E、F，每一块只做最薄、最简版本，把整条链路打通，形成闭环。

> 重点：所有契约，在第一轮就全部跑一遍验证。不需要写复杂清洗、不需要复杂模型，只用少量样例数据，走一遍：数据读取→清洗→打标签→样本切分→训练→打包模型→API 预测，整条链路跑通。

- **第 2 轮 L2**：各个模块慢慢 "加厚"，补齐清洗逻辑、补齐规则，和冻结基线对账，核对结果，保证计算结果和基线一致。
- **第 3 轮 L3**：按需扩展，把功能做丰富，增加更多特征、更多校验、更多指标。

**它的核心优势一句话：任何时候停下来，手里都有一个可以演示、可以跑通的完整系统。**

哪怕只做完第一轮，它不是半成品碎片，而是一套完整的流水线，只是功能简单。

契约、接口、依赖关系，在最早的第一轮就验证，接口错了，很早就发现，不会等到写了一大堆代码之后才翻车。

架构先行，是先定好整套架构规则，而不是一次性写完所有代码；优先搭能跑通的极简骨架，闭环验证契约，再慢慢加厚功能，这就是迭代增量。

### 2. 名词解释

| 名词 | 英文 / 出处 | 说的是什么 |
|---|---|---|
| 迭代增量开发 | Iterative and Incremental Development | 一轮一轮来，每轮都交付能用的增量 |
| 行走的骨架 | Walking Skeleton（Alistair Cockburn） | 先做一个极简但端到端连通的系统；主要组件都在、都连上，功能最少 |
| 曳光弹 | Tracer Bullet（《程序员修炼之道》） | 先打一条贯穿所有层的真实代码路径，确认能击中目标，再补全 |
| 最小可行产品 | MVP | 面向用户的最小可用版本 |
| 垂直切片 | Vertical Slice | 一个功能从界面到存储一刀切到底 |
| 先能跑，再做对，再做快 | Make it work, make it right, make it fast（Kent Beck） | 优先级口诀 |

### 3. 六阶段 × 三层级

六个阶段 A-F，每一个阶段都拆成 L1 骨架、L2 加厚、L3 扩展三层；并且严格遵守两条开发规则：先横着做完一行，再做下一行；同一行里面按 A→F 顺序走。

#### 3.1 解释三层

- **L1 骨架（第 1 轮）**：闭环必需的最薄一层，必做 目标：能跑通整条端到端链路，也就是我们前面说的行走骨架、曳光弹。不追求功能多、模型好，只保证链路通、契约对。
- **L2 加厚（第 2 轮）**：让结果 "做对" 核心动作就是对账。拿数据和冻结数据集比对，输出结果和第一轮跑出来的冻结基线比对，保证逻辑没错、计算结果一致；补齐清洗规则、更多模型、指标。
- **L3 扩展（第 3 轮）**：让系统 "做宽" 目标：丰富能力、适配更多场景、优化工程能力，支持更多数据源、更多模型调参、上线监控看板、权限、压测、告警这些。

两条开发规则，一定要记住：

1. 先横着把一行做完，再做下一行：也就是先做完全部模块的 L1 骨架（A-F 全部 L1），闭环跑通之后，再回头统一做全部模块的 L2 加厚；L2 全部对账完成，再做全部 L3 扩展。
2. 同一行内部，按 A→F 顺序：A 底层设施 → B 数据准备 → C 训练评估 → D 发布注册 → E 服务观测 → F 监控回流，不能跳着做。

#### 3.2 A-F 各个阶段

**A 底层设施（地基）**

- L1：第 0 轮就搞定基础底层配置，是整个项目的地基
- L2：随着项目生长，补齐配置、分层契约、文档说明
- L3：搭流水线，提交代码自动校验（pre-commit），代码仓库自动化流程

**B 数据准备（数据原料）**

- L1：最简的数据构建清单 manifest、基础切分检查，能跑出一份冻结数据集
- L2：完善数据源、清洗规则、标签规则，接入 DVC 做数据版本，开始第一轮对账
- L3：支持标准构建器，兼容 Excel、CSV 这类更多类型数据源

**C 训练与评估（模型训练）**

- L1：最简单的逻辑回归、基础指标、特征规范，能完成训练，产出模型
- L2：多种评估口径、完整指标，增加 LightGBM、随机森林、HGB 等模型，第二、第三次对账
- L3：样本切片分析、模型校准、模型卡、超参数调参，精细化实验

**D 发布与注册（模型打包上线准备）**

- L1：基础 ModelBundle 打包，导出模型包
- L2：权限控制、MLflow 模型注册、版本发布流程
- L3：模型封控、状态机、回滚机制、审计记录，线上出问题可以回退到老版本

**E 服务与观测（线上推理服务）**

- L1：预测接口、预测结果存储，最简推理服务
- L2：监控指标、热加载、兼容 MLflow 来源模型
- L3：Docker 容器部署、Grafana 监控面板、压测工具 Locust、鉴权登录

**F 监控与回流（线上效果跟踪）**

- L1：流量回放、PSI 分布漂移监控、基础重训手册
- L2：标签回流、线上效果验证、对照实验
- L3：批量导入、策略管理、调度告警，自动监控预警

**前端页面配套**

- L1：最简单页面，单条样本评估、模型信息查看
- L2：批量评估、特征贡献因素图（SHAP 图表）
- L3：文件上传、大盘监控、登录审批

阶段执行的正确顺序：

第一轮：A (L1) → B (L1) → C (L1) → D (L1) → E (L1) → F (L1)，横向走完 L1，整条链路跑通闭环；

第二轮：A (L2) → B (L2) → C (L2) → D (L2) → E (L2) → F (L2)，横向走完 L2，全部对账，保证结果正确；

第三轮：A (L3) → B (L3) → C (L3) → D (L3) → E (L3) → F (L3)，横向走完 L3，扩展能力。

口诀：先能跑（L1），再做对（L2），再做快、做全（L3）